In [1]:
import robotic as ry
import numpy as np
import random
import time 
import WayTu_RAI.model_utils as mutils
import WayTu_RAI.main as main
from WayTu_RAI.GenerateEnvironment import GenerateEnvironment, PlacementError



ry.params_add({'physx/motorKp': 10000., 
               'physx/motorK  d': 1000., 
               'physx/angularDamping': 10., 
               'physx/defaultFriction': 1000.})

ry.params_add({'botsim/engine': 'physx'}) #makes a big difference!
ry.params_add({'physx/multibody': True}) #makes a big difference!
ry.params_print()

print(ry.__version__)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
0.1.10physx/motorKp: 
10000,
physx/motorK  d: 1000,
physx/angularDamping: 10,
physx/defaultFriction: 1000,
botsim/engine: physx,
physx/multibody

In [2]:
SEED = 48

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
cfg = {
    "mode" : "train", 
        "num-tools" : 1,
        "num-obj-points": 512,
        "label-list-all" : ["lifting-platform", "minigolf-platform", "hammering-platform",
                      "hammer", "spatula", "L-ruler", "reaching-platform", "pouring-platform", "pitcher"],
        # "model-name" :  "waytu_unified_model_hammering_v25_best.pth",
        "model-name" : "waytu_unified_model_minigolf_v22_best.pth",
        "feature-extractor-path" : "small-pointner-encoder-distractor_best.pth",
        "feature-size" : 128,
        "num-trials" : 1, 
        "tool-type" : "additional",
        # "task" : "hammering", 
        # "task" : "minigolf",
        "task" : "pouring",
        "dataset-save-path" : "test-dataset",
        "area-middle": {
            "min" : [-0.30 , 0.15, 0.060],
            "max" : [ 0.30 , 0.45, 0.065]
            },
        "area-negative" : {
            "min": [-0.40 , 0.15, 0.050],
            "max": [ -0.30 , 0.25, 0.060],
            },
        "area-positive" : {
            "min": [0.30 , 0.15, 0.050],
            "max": [ 0.40 , 0.25, 0.060],
            },
        "cameras" : ["camera1", "camera2", "camera3" ],
    
}

In [4]:
total_count = 0 
num_trials = cfg["num-trials"]
for trial in range(num_trials):
    total_count += 1    
    try:
        environment = GenerateEnvironment(cfg)
        environment.generate_environment()

        env_pcl = environment.point_clouds_labels

        # Extract the point cloud and labels
        env_pc = env_pcl[:, :3]
        env_label = env_pcl[:, 3]

        # Select a random tool
        selected_tool, tool_label = environment.select_random_tool()
        # Collect samples for a tool 
        # selected_tool, tool_label = "ball", 7
        print(f"Selected tool: {selected_tool}, label: {tool_label}")
        tool_mask = env_label == tool_label 
        tool_pc = env_pc[tool_mask]

        # Get platform points
        environment_mask = env_label < 3
        environment_pc = env_pc[environment_mask] 

        other_tools = [item for item in environment.tool_objs if item != selected_tool]
        time.sleep(15.0)
        waypoints_static = environment.env.get_waypoints_static(environment.C)
        waypoints = environment.set_waypoints(
                        waypoint= waypoints_static,
                        selected_tool=selected_tool,
                        other_tools=other_tools
                    )
        
        score = mutils.manipulation_with_komo(environment.C)
        environment.C.view()
    except PlacementError as e:
        print(f"[!] Skipping sample due to error: {e}")
        continue

{'name': 'table', 'ID': 1, 'rel': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0], 'shape': 'ssBox', 'size': [2.5, 2.5, 0.1, 0.02], 'color': [0.3, 0.3, 0.3], 'contact': 1, 'logical': {}, 'friction': 0.1, 'X': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0]}
table threshold:  0.655
camera height:  [0.   0.9  1.22]
{'min': array([-0.30086512,  0.24346405,  0.53151125]), 'max': array([-0.14086512,  0.48346405,  0.79151125]), 'center': array([-0.22086512,  0.36346405,  0.66151125]), 'quaternion': array([ 0.7262643 ,  0.        ,  0.        , -0.68741557]), 'rotation_z': -86.85175123239253}
(6078, 3)
(6078, 3)
obj name: pitcher, label: 8
AAA [<bound method PouringEnvironment.getPlatform of <WayTu_RAI.Environments.PouringEnvironment.PouringEnvironment object at 0x72cc980ca410>>]
Before Update [0.9790554301808645, 0.0, 0.0, 0.20359387179716956]
After Update [0.9790554301808645, 0.0, 0.0, 0.20359387179716956]
{'min': array([0.19927183, 0.1061985 , 0.60165642]), 'max': array([0.43927183, 0.3461985 , 0.80165642]), 'cen